<a href="https://colab.research.google.com/github/mkwonghsni/bengkel_daml_2026/blob/main/hari_kedua/torch_nn_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


import torch
from torch import nn
from torch.utils.data import DataLoader


import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor

from mlxtend.plotting import plot_confusion_matrix

try:
    from torchinfo import summary
    from torchmetrics import ConfusionMatrix
except ModuleNotFoundError:
    !pip install -q torchinfo torchmetrics
    from torchinfo import summary
    from torchmetrics import ConfusionMatrix

# Tetapan

In [ ]:
# random number generator
rng = np.random.default_rng()

## Soalan: Apakah itu batching? Mengapa pilih *32* sebagai saiz batch?

In [ ]:
# saiz setiap batch data
SAIZ_BATCH = 32
print(f'Saiz batch: {SAIZ_BATCH}')

## Tip: Runtime type

In [ ]:
# tetapkan peranti yang akan digunakan: cpu vs cuda
PERANTI = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Peranti: {PERANTI}')

## Ulangkaji: Apakah itu epoch?

In [ ]:
EPOCHS = 10 if PERANTI == "cuda" else 3
print(f'Epochs = {EPOCHS}')

# Fungsi
## Ulangkaji: Kita mencipta fungsi (custom function) untuk mengelakkan kod berulang.

In [ ]:
def blok_conv(masuk, keluar):
    '''
    Convolutional block yang mengandungi 2 lapisan Conv2d, 2 lapisan ReLU, dan 1 lapisan MaxPool2d.
    Conv2d -> ReLU -> Conv2d -> ReLU -> MaxPool2d
    '''
    return nn.Sequential(
        nn.Conv2d(in_channels=masuk, out_channels=keluar, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=keluar, out_channels=keluar, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
        )

In [ ]:
def blok_klasifikasi(masuk, bil_kelas):
    '''
    Blok klasifikasi yang mengandungi 1 lapisan Flatten dan 1 lapisan Linear.
    Flatten -> Linear
    '''
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=masuk, out_features=bil_kelas)
        )

In [ ]:
def melatih_model_nn(model, dataloader, loss_fx, optimizer, peranti):
    '''
    Proses melatih model neural network dengan data latihan.
    Data disampaikan dengan menggunakan dataloader.
    Model dan data ditempatkan dalam peranti.
    '''
    loss = 0.

    model.to(peranti)

    model.train()

    for batch, (imej, label) in enumerate(dataloader):
        imej, label = imej.to(peranti), label.to(peranti)
        y = model(imej)
        loss_batch = loss_fx(y, label)
        loss += loss_batch
        optimizer.zero_grad()
        loss_batch.backward()
        optimizer.step()

    loss /= len(dataloader)
    print(f'Loss latihan: {loss:.4f}')

In [ ]:
def menguji_model_nn(model, dataloader, loss_fx, peranti):
    '''
    Proses menguji model neural network dengan data ujian.
    Data disampaikan dengan menggunakan dataloader.
    Model dan data ditempatkan dalam peranti.
    '''
    loss = 0.

    model.to(peranti)

    model.eval()

    with torch.inference_mode():
        for imej, label in dataloader:
            imej, label = imej.to(peranti), label.to(peranti)
            y = model(imej)
            loss += loss_fx(y, label)

        loss /= len(dataloader)

        print(f'Loss ujian: {loss:.4f}')

# Menyediakan Data
1. Download data (imej) untuk latihan dan ujian
2. Semak bilangan data
3. Tetapkan label data
4. Paparkan data (imej) secara rawak

## Tip: Data MNIST juga boleh dimuat turun ke dalam PC dan seterusnya dimuat naik ke dalam Colab

In [ ]:
data_latihan = datasets.MNIST(root="data", train=True, transform=ToTensor(), download=True)
data_ujian = datasets.MNIST(root="data", train=False, transform=ToTensor(), download=True)

In [ ]:
len(data_latihan), len(data_ujian)

## Soalan: Apakah data kita sebenarnya?
***Tip:*** Structured vs unstructured data

## Tip: Jika MNIST dimuat turun / naik secara manual, kod berikut perlu diubah

In [ ]:
label_digit = data_latihan.classes
label_digit

In [ ]:
bil = torch.bincount(data_latihan.targets)

print("Bilangan data (imej) dalam data latihan")
for _, (l, b) in enumerate(zip(label_digit, bil)):
    print(f"{l}: {b}")

In [ ]:
bil = torch.bincount(data_ujian.targets)

print("Bilangan data (imej) dalam data ujian")
for _, (l, b) in enumerate(zip(label_digit, bil)):
    print(f"{l}: {b}")

## Tip: Langkah-langkah yang kita lakukan di atas bersamaan dengan EDA
## Latihan: Cuba paparkan imej (secara rawak) beserta labelnya dalam grid 4×4

# Analisis Data
1. Bina dataloader
2. Bina model NN
3. Melatih model
4. Menilai model dengan menggunakan metrik Confusion Matrix

## Perhatikan: Data tidak dimasukkan sekaligus → dihantar secara batch
## Ulangkaji: Mengapa kita menggunakan saiz batch 32?
## Ulangkaji: Bagaimana / di mana kita boleh mendapatkan dokumentasi untuk method `DataLoader()`

## Soalan: Bagaimana rupa `model` kita? Adakah model ini masih sama seperti linear_nn sebelum ini?

***Tip:*** CNN Explainer

In [ ]:
model = nn.Sequential(
    blok_conv(masuk=1, keluar=8),
    blok_conv(masuk=8, keluar=8),
    blok_klasifikasi(masuk=8*7*7, bil_kelas=10)
)

## Satu lagi cara untuk memaparkan struktur `model` kita

In [ ]:
summary(model, input_size=(SAIZ_BATCH, 1, 28, 28))

## Ulangkaji: Apakah itu loss? Apakah fungsi optimizer?
## Perhatikan: Loss dan optimizer telah berubah. Mengapa?

In [ ]:
loss_fx = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Ulangkaji: Apakah itu epoch?
## Perhatikan: Perubahan pada loss latihan dan loss ujian. Apakah yang boleh disimpulkan daripada perubahan loss latihan dan loss ujian?

In [ ]:
%%time
for epoch in range(EPOCHS):
    print(f'Epoch: {epoch+1}\n---------')
    melatih_model_nn(model, dataloader_latihan, loss_fx, optimizer, PERANTI)
    menguji_model_nn(model, dataloader_ujian, loss_fx, PERANTI)
    print()

## Soalan: Adakah loss latihan dan loss ujian menunjukkan corak yang sama?
***Tip:*** Overfitting vs underfitting
## Perhatikan: Label sebenar vs ramalan

## Soalan: Bagaimana prestasi `model_nn`?

In [ ]:
y_ramal = []

model.eval()
with torch.inference_mode():
    for imej, label in dataloader_ujian:
        y = model(imej)
        y_ramal.append(y.argmax(dim=1))

y_ramal_tensor = torch.cat(y_ramal)

## Soalan: Bagaimana mentafsirkan prestasi model_nn dengan menggunakan confusion matrix?
## Perhatikan: Apakah perbezaan antara method `ConfusionMatrix()` daripada library `PyTorch` dengan `confusion_matrix()` daripada library `Sklearn`?

In [ ]:
confmat_tensor = ConfusionMatrix(task="multiclass", num_classes=len(label_digit))(preds=y_ramal_tensor, target=data_ujian.targets)

# plot_confusion_matrix(conf_mat=confmat_tensor.numpy(), class_names=label_digit)
plot_confusion_matrix(conf_mat=confmat_tensor.numpy(), class_names=label_digit, show_absolute=False, show_normed=True)
plt.show()

## Ulangkaji: Apakah nilai parameter `model` kita?

In [ ]:
model.state_dict()